# Annotated somatic variants with tumor tissue of origin

## cBioPortal data annotated using OncoTree ontology in sample level information

In [ ]:
import os
import re
import json
import pandas as pd
import hashlib
import csv
import numpy as np
#loading clinical data file of all samples in 213 non-redundant studies in cBioPortal as of May 2023 (extracted using cBioPortalR):
cbio_sampleinfo=pd.read_csv("PATHtocBioPortal/cBio_all_clinical_fromRStudio_6-14-23.txt", sep="\t")
cbio_sampleinfo_cancertypedetailed=cbio_sampleinfo[cbio_sampleinfo["clinicalAttributeId"].str.contains("CANCER_TYPE_DETAILED")==True]
cbio_sampleinfo_cancertypedetailed_uniquesamplekey=cbio_sampleinfo_cancertypedetailed.drop_duplicates(subset=['uniqueSampleKey'])

#load OncoTreeAPI file:
!curl -X GET --header 'Accept: text/plain' 'https://oncotree.mskcc.org:443/api/tumor_types.txt?version=oncotree_latest_stable' -o PATHtocBioPortal/OncoTree_API_tumortypes_oncotree_latest_stable_version_4-5-24.txt 

OncoTreeAPI_output=pd.read_csv("PATHtocBioPortal/OncoTree_API_tumortypes_oncotree_latest_stable_version_4-5-24.txt", sep="\t")
OncoTreeAPI_output_fill0=OncoTreeAPI_output.fillna('N/A')

cbio_sampleinfo_cancertypedetailed_uniquesamplekey[["parent_tissue_code","count_parents_in_OncoTree","unique_count_parents_in_OncoTree"]]=['NaN','NaN','NaN']
for index1, row1 in cbio_sampleinfo_cancertypedetailed_uniquesamplekey.iterrows():
    oncotree_string=cbio_sampleinfo_cancertypedetailed_uniquesamplekey.loc[index1,'value']
    #for spot check, print(oncotree_string)
    count=0
    uniquecount=0
    parent_tissue_list=[]#to output list of parent tissue options if more than 1 for same oncotree string
    for index2, row2 in OncoTreeAPI_output_fill0.iterrows():
        for j in OncoTreeAPI_output_fill0.columns:
            if oncotree_string in OncoTreeAPI_output_fill0.loc[index2,j]:
                parent_tissue=OncoTreeAPI_output_fill0.loc[index2,'level_1']
                count=count+1
                parent_tissue_list.append(parent_tissue)
                #print(parent_tissue)
                #print(parent_tissue_list)
                
    if (len(parent_tissue_list) != 0):
        unique_parent_list=[]
        for j in (parent_tissue_list):
            if j not in unique_parent_list:
                unique_parent_list.append(j)
                uniquecount=uniquecount+1
    
    cbio_sampleinfo_cancertypedetailed_uniquesamplekey.loc[index1,'unique_count_parents_in_OncoTree']=uniquecount          
    cbio_sampleinfo_cancertypedetailed_uniquesamplekey.loc[index1,'count_parents_in_OncoTree']=count
    cbio_sampleinfo_cancertypedetailed_uniquesamplekey.loc[index1,'parent_tissue_code']=parent_tissue
    parent_tissue="Cancer Type Detailed not in OncoTree API file"
    #for spot check, 
    #print(cbio_sampleinfo_cancertypedetailed_uniquesamplekey.loc[index1,'unique_count_parents_in_OncoTree'])

#manually correcting the strings in .txt file for the 3661 samples without OT annotations #10-24-24 
#loading that .txt file and using new string to get OT parent for 3661 samples
#then delete these '0' in unique count column samples from the larger file and concat the newly annotated 3661 samples instead

In [210]:
pddfnotinOT_manuallycorrected=pd.read_csv("cBioPortal/cbio_manuallycorrected_CANCERTYPEDETAILED_notin_OncoTree_UTF-8.txt", sep="\t")#, encoding_errors='replace')


In [211]:
cbio_sampleinfo_0_parentOT_manuallycorrected=cbio_sampleinfo_0_parentOT.reset_index().set_index("value").join(pddfnotinOT_manuallycorrected.set_index("Cancer_Type_Detailed"), how='left', rsuffix="manualfile")


In [212]:
cbio_sampleinfo_0_parentOT_manuallycorrected=cbio_sampleinfo_0_parentOT_manuallycorrected.reset_index().set_index("index")

In [213]:
cbio_sampleinfo_0_parentOT_manuallycorrected_qcnan=cbio_sampleinfo_0_parentOT_manuallycorrected_fillna0[cbio_sampleinfo_0_parentOT_manuallycorrected_fillna0["Manually_corrected_Cancer_Type_Detailed"]==0]


In [112]:
cbio_sampleinfo_0_parentOT_manuallycorrected_qcnan["Manually_corrected_Cancer_Type_Detailed"]='Chromophobe Renal Cell Carcinoma'


/var/folders/5h/8jz2ndj504sb9gtp0mt16p680000gn/T/ipykernel_66397/2861390432.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cbio_sampleinfo_0_parentOT_manuallycorrected_qcnan["Manually_corrected_Cancer_Type_Detailed"]='Chromophobe Renal Cell Carcinoma'


In [215]:
cbio_sampleinfo_0_parentOT_manuallycorrected_again=cbio_sampleinfo_0_parentOT_manuallycorrected[~cbio_sampleinfo_0_parentOT_manuallycorrected["Manually_corrected_Cancer_Type_Detailed"].isna()==True]
cbio_sampleinfo_0_parentOT_manuallycorrected_again=pd.concat([cbio_sampleinfo_0_parentOT_manuallycorrected_again,cbio_sampleinfo_0_parentOT_manuallycorrected_qcnan])

In [132]:
cbio_sampleinfo_0_parentOT_manuallycorrected_fillna0=cbio_sampleinfo_0_parentOT_manuallycorrected.fillna(0)
for index1, row1 in cbio_sampleinfo_0_parentOT_manuallycorrected_fillna0.iterrows():
                
    if (cbio_sampleinfo_0_parentOT_manuallycorrected_fillna0.loc[index1, 'Manually_corrected_Cancer_Type_Detailed']=='0'):
        colindex=cbio_sampleinfo_0_parentOT_manuallycorrected_fillna0.columns.get_loc('Manually_corrected_Cancer_Type_Detailed')
        cbio_sampleinfo_0_parentOT_manuallycorrected_fillna0.loc[index1, colindex]='Chromophobe Renal Cell Carcinoma'     
    

In [216]:
cbio_sampleinfo_0_parentOT_manuallycorrected_again["Manually_corrected_Cancer_Type_Detailed"].value_counts()

Hepatocellular Carcinoma                                         1139
Esophagogastric Adenocarcinoma                                    866
Pilocytic Astrocytoma                                             105
Chromophobe Renal Cell Carcinoma                                  103
Low-Grade Glioma                                                   94
                                                                 ... 
Mucinous Adenocarcinoma Lymph Node                                  1
Nephroblastomatosis                                                 1
Nested stromal epithelial tumor of the liver                        1
Non-Seminomatous Germ Cell Tumor, predominantly yolk sac type       1
Mucoepidermoid Carcinoma of the Lung                                1
Name: Manually_corrected_Cancer_Type_Detailed, Length: 89, dtype: int64

In [217]:
cbio_sampleinfo_0_parentOT_manuallycorrected_again_qcnan=cbio_sampleinfo_0_parentOT_manuallycorrected_again[cbio_sampleinfo_0_parentOT_manuallycorrected_again["Manually_corrected_Cancer_Type_Detailed"].isna()==True]


In [218]:
cbio_sampleinfo_0_parentOT_manuallycorrected_again_qcnan

,level_0,uniqueSampleKey,unique_count_parents_in_OncoTree,parent_tissue_code_list,count_parents_in_OncoTree,parent_tissue_code,uniquePatientKey,sampleId,patientId,studyId,clinicalAttributeId,Manually_corrected_Cancer_Type_Detailed,value,Why
index,,,,,,,,,,,,,,


In [219]:
cbio_sampleinfo_0_parentOT_manuallycorrected=cbio_sampleinfo_0_parentOT_manuallycorrected_again.filter(items=['level_0', 'uniqueSampleKey', 'uniquePatientKey', 'sampleId', 'patientId', 'studyId', 'clinicalAttributeId', 'Manually_corrected_Cancer_Type_Detailed', 'value', 'Why'])


#rerun annotating with parent tissue type:

In [ ]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)
cbio_sampleinfo_0_parentOT_manuallycorrected.insert(1,"parent_tissue_code",'NaN')
cbio_sampleinfo_0_parentOT_manuallycorrected.insert(1,"count_parents_in_OncoTree",'NaN')
cbio_sampleinfo_0_parentOT_manuallycorrected.insert(1,"parent_tissue_code_list",'NaN')
cbio_sampleinfo_0_parentOT_manuallycorrected.insert(1,"unique_count_parents_in_OncoTree",'NaN')


for index1, row1 in cbio_sampleinfo_0_parentOT_manuallycorrected.iterrows():
    oncotree_string=cbio_sampleinfo_0_parentOT_manuallycorrected.loc[index1,'Manually_corrected_Cancer_Type_Detailed']
    #print(oncotree_string)
    count=0
    uniquecount=0
    parent_tissue_list=[]#to output list of parent tissue options if more than 1
    for index2, row2 in OncoTreeAPI_output_fill0.iterrows():
        for j in OncoTreeAPI_output_fill0.columns:
            if oncotree_string in OncoTreeAPI_output_fill0.loc[index2,j]:
                parent_tissue=OncoTreeAPI_output_fill0.loc[index2,'level_1']
                count=count+1
                parent_tissue_list.append(parent_tissue)
    #print(parent_tissue)
                
    if (len(parent_tissue_list) != 0):
        unique_parent_list=[]
        for j in (parent_tissue_list):
            if j not in unique_parent_list:
                unique_parent_list.append(j)
                uniquecount=uniquecount+1
    
    colindexuniquecount=cbio_sampleinfo_0_parentOT_manuallycorrected.columns.get_loc('unique_count_parents_in_OncoTree')
    cbio_sampleinfo_0_parentOT_manuallycorrected.at[index1,colindexuniquecount]=uniquecount          
    colindexlist=cbio_sampleinfo_0_parentOT_manuallycorrected.columns.get_loc('parent_tissue_code_list')
    cbio_sampleinfo_0_parentOT_manuallycorrected.at[index1,colindexlist]=str(parent_tissue_list)
    colindexcount=cbio_sampleinfo_0_parentOT_manuallycorrected.columns.get_loc('count_parents_in_OncoTree')
    cbio_sampleinfo_0_parentOT_manuallycorrected.at[index1,colindexcount]=count
    colindex=cbio_sampleinfo_0_parentOT_manuallycorrected.columns.get_loc('parent_tissue_code')
    cbio_sampleinfo_0_parentOT_manuallycorrected.at[index1,colindex]=parent_tissue
    parent_tissue="Cancer Type Detailed not in OncoTree API file"
    print(row1)
end_time_block2=time.asctime(time.localtime())
print(end_time_block2)

In [243]:
# first remove all 3661 with '0' parents in OT
cbio_sampleinfo_cancertypedetailed_uniquesamplekey_no0inOT=cbio_sampleinfo_cancertypedetailed_uniquesamplekey[cbio_sampleinfo_cancertypedetailed_uniquesamplekey["unique_count_parents_in_OncoTree"]!=0]


In [247]:
# then concat this file to it
cbio_sampleinfo_cancertypedetailed_uniquesamplekey_concat=pd.concat([cbio_sampleinfo_cancertypedetailed_uniquesamplekey_no0inOT, cbio_sampleinfo_0_parentOT_manuallycorrected_renamecolumns])


#Annotate processed OLO datasets per gene

In [253]:
# then annotate all vars in cbio
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)
clinvarCategories=pd.read_csv("ClinVarCategories_comparewithCOSMIC_10-30-23.txt", sep="\t", header=None).set_index(0)
Hypermutators=pd.read_csv("cbio_sampleinfo_tmb_gt10_12-14-23.txt", sep="\t")
Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index

cbio=pd.DataFrame()

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        
        clinvar_cbio_tocompare=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_cbio_tocompare_8-16-24.txt", sep="\t")
        Variation_interpretation_toexclude = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic", "Conflicting interpretations of pathogenicity greater than or equal to 75"]
        clinvar_cbio_tocompare_VUSLBB=clinvar_cbio_tocompare[~clinvar_cbio_tocompare["GL_first_Description"].isin(Variation_interpretation_toexclude)]
        #print("shared count")
        #print(len(clinvar_cbio_tocompare))
        #print("shared that are VUSLBB")
        #print(len(clinvar_cbio_tocompare_VUSLBB))
        cbio_VEP=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_8-16-24.txt", sep="\t")
        #print("cbio VEP len")
        cbioCount1=len(cbio_VEP)
        #print(len(cbio_VEP))
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"] #8-23-24 #, ">=50bp_indel"]
        cbio_VEP=cbio_VEP[~cbio_VEP["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        #print("cbio VEP len after filter VEP categories")
        #print(len(cbio_VEP))
        cbioCount2=len(cbio_VEP)
        Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index
        #print("with hypermutators count QC same as above")
        totalbeforehypermutatorfilter=len(cbio_VEP)
        #print(len(cbio_VEP))
        cbio_VEP_nohypermutator=cbio_VEP[~cbio_VEP["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        cbio_VEP=cbio_VEP_nohypermutator
        #print("without hypermutators count")
        #print(len(cbio_VEP))
        #print("% vars lost to hypermutator filter=#total-#afterfilter/#total")
        #print(((totalbeforehypermutatorfilter-len(cbio_VEP))*100)/totalbeforehypermutatorfilter)
        removetheseVUSLBB=clinvar_cbio_tocompare_VUSLBB["@id"]
        cbio_VEP_noVUSLBB=cbio_VEP[~cbio_VEP["@id"].isin(removetheseVUSLBB)]
        #print("without VUSLBB")
        #print(len(cbio_VEP_noVUSLBB))
        #print("# variants lost after VUSLBB filter")
        #print(len(cbio_VEP)-len(cbio_VEP_noVUSLBB))
        #print("% vars lost to VUSLBB filter=#total-#afterfilter/#total")
        #print(((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP))
        cbio_VEP_cancertype=cbio_VEP_noVUSLBB
        print(len(cbio_VEP_cancertype))
        #cbio=pd.concat([cbio, cbio_VEP_cancertype])
        
        cbio_VEP_cancertype_OT=cbio_VEP_cancertype.set_index("uniqueSampleKey").join(cbio_sampleinfo_cancertypedetailed_uniquesamplekey_concat.set_index("uniqueSampleKey"), how='left', rsuffix="cbio_sample_clinical_file")
        #print(len(cbio_VEP_cancertype_OT))
        cbio_VEP_cancertype_OT_nan=cbio_VEP_cancertype_OT[cbio_VEP_cancertype_OT["unique_count_parents_in_OncoTree"].isna()==True]
        #print(len(cbio_VEP_cancertype_OT_nan))
        #display(cbio_VEP_cancertype_OT_nan)
        #display(cbio_VEP_cancertype_OT["unique_count_parents_in_OncoTree"].value_counts())
        cbio_VEP_cancertype_OT_noOT=cbio_VEP_cancertype_OT[cbio_VEP_cancertype_OT["unique_count_parents_in_OncoTree"]==0]
        #display(cbio_VEP_cancertype_OT_noOT["Manually_corrected_Cancer_Type_Detailed"].value_counts())
        display(cbio_VEP_cancertype_OT["parent_tissue_code"].value_counts())                  
        cbio_VEP_cancertype_OT.to_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_OTtissue_10-24-24.txt", sep="\t")
        
        # to annotate the CDS location file and save that too:
        cbio_VEP_CDS=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_CDSLocationfiltered_8-29-24.txt", sep="\t")
        print(len(cbio_VEP_CDS))
        cbio_VEP_CDS_OT=cbio_VEP_CDS.set_index("uniqueSampleKey").join(cbio_sampleinfo_cancertypedetailed_uniquesamplekey_concat.set_index("uniqueSampleKey"), how='left', rsuffix="cbio_sample_clinical_file")
        print(len(cbio_VEP_CDS_OT))
        cbio_VEP_CDS_OT_nan=cbio_VEP_CDS_OT[cbio_VEP_CDS_OT["unique_count_parents_in_OncoTree"].isna()==True]
        print(len(cbio_VEP_CDS_OT_nan))
        #display(cbio_VEP_cancertype_OT_nan)
        #display(cbio_VEP_cancertype_OT["unique_count_parents_in_OncoTree"].value_counts())
        #cbio_VEP_CDS_OT_noOT=cbio_VEP_cancertype_OT[cbio_VEP_cancertype_OT["unique_count_parents_in_OncoTree"]==0]
        #display(cbio_VEP_cancertype_OT_noOT["Manually_corrected_Cancer_Type_Detailed"].value_counts())
        display(cbio_VEP_CDS_OT["parent_tissue_code"].value_counts())                  
        cbio_VEP_CDS_OT.to_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_CDSLocationfiltered_OTtissue_10-24-24.txt", sep="\t")
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)


Thu Oct 24 16:02:49 2024
131


Myeloid (MYELOID)                  77
Lymphoid (LYMPH)                   10
Lung (LUNG)                         9
Kidney (KIDNEY)                     6
Pancreas (PANCREAS)                 4
Bowel (BOWEL)                       4
Liver (LIVER)                       4
CNS/Brain (BRAIN)                   3
Esophagus/Stomach (STOMACH)         3
Bladder/Urinary Tract (BLADDER)     2
Bone (BONE)                         2
Breast (BREAST)                     1
Head and Neck (HEAD_NECK)           1
Name: parent_tissue_code, dtype: int64

131
131
5


Myeloid (MYELOID)                  77
Lymphoid (LYMPH)                   10
Lung (LUNG)                         9
Kidney (KIDNEY)                     6
Pancreas (PANCREAS)                 4
Bowel (BOWEL)                       4
Liver (LIVER)                       4
CNS/Brain (BRAIN)                   3
Esophagus/Stomach (STOMACH)         3
Bladder/Urinary Tract (BLADDER)     2
Bone (BONE)                         2
Breast (BREAST)                     1
Head and Neck (HEAD_NECK)           1
Name: parent_tissue_code, dtype: int64

Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1
634


...

Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/ATM
4060


Bowel (BOWEL)                                    3443
Lung (LUNG)                                       143
Esophagus/Stomach (STOMACH)                       102
Prostate (PROSTATE)                                79
Liver (LIVER)                                      78
Ampulla of Vater (AMPULLA_OF_VATER)                42
Biliary Tract (BILIARY_TRACT)                      35
Pancreas (PANCREAS)                                21
Other (OTHER)                                      14
Skin (SKIN)                                        11
Breast (BREAST)                                    11
Bladder/Urinary Tract (BLADDER)                     7
Uterus (UTERUS)                                     7
Ovary/Fallopian Tube (OVARY)                        6
Head and Neck (HEAD_NECK)                           6
CNS/Brain (BRAIN)                                   5
Kidney (KIDNEY)                                     5
Thyroid (THYROID)                                   5
Vulva/Vagina (VULVA)        

4054
4054
26


Bowel (BOWEL)                                    3440
Lung (LUNG)                                       143
Esophagus/Stomach (STOMACH)                       102
Liver (LIVER)                                      78
Prostate (PROSTATE)                                78
Ampulla of Vater (AMPULLA_OF_VATER)                42
Biliary Tract (BILIARY_TRACT)                      35
Pancreas (PANCREAS)                                21
Other (OTHER)                                      14
Skin (SKIN)                                        11
Breast (BREAST)                                    11
Bladder/Urinary Tract (BLADDER)                     7
Uterus (UTERUS)                                     7
Head and Neck (HEAD_NECK)                           6
Ovary/Fallopian Tube (OVARY)                        5
Kidney (KIDNEY)                                     5
Thyroid (THYROID)                                   5
CNS/Brain (BRAIN)                                   4
Vulva/Vagina (VULVA)        

Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/APC
Thu Oct 24 16:05:23 2024


## For COSMIC data, convert NCIt ontology to OncoTree

In [159]:
!python ontology_to_ontology_mapping_tool.py --source-file Cosmic_Classification_v98_GRCh38_11-20-24.tsv --target-file Cosmic_Classification_v98_GRCh38_11-20-24_O2OMAP_OncoTree.txt --source-code NCIT_CODE --target-code ONCOTREE_CODE


Mapped the NCIT_CODE to ONCOTREE_CODE..

The ontology mappings are written to: Cosmic_Classification_v98_GRCh38_11-20-24_O2OMAP_OncoTree.txt
The mapping summary is written to : Cosmic_Classification_v98_GRCh38_11-20-24_O2OMAP_OncoTree_summary.html


#Then annotate samples in COSMIC with parent tissue of origin based on OncoTree ontology

In [ ]:
# annotate the unique set COSMIC_classification_OT with parent tissue type:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)
COSMIC_classification_OT.insert(1,"parent_tissue_code",'NaN')
COSMIC_classification_OT.insert(1,"count_parents_in_OncoTree",'NaN')
COSMIC_classification_OT.insert(1,"parent_tissue_code_list",'NaN')
COSMIC_classification_OT.insert(1,"unique_count_parents_in_OncoTree",'NaN')


for index1, row1 in COSMIC_classification_OT.iterrows():
    oncotree_string=str(COSMIC_classification_OT.loc[index1,'ONCOTREE_CODE']).split(", ")
    print(oncotree_string)
    count=0
    uniquecount=0
    parent_tissue_list=[]#to output list of parent tissue options if more than 1
    for string in range(len(oncotree_string)):
        print(f"({oncotree_string[string]})")
        for index2, row2 in OncoTreeAPI_output_fill0.iterrows():
            for j in OncoTreeAPI_output_fill0.columns:
                if f"({oncotree_string[string]})" in OncoTreeAPI_output_fill0.loc[index2,j]:
                    
                    print(string)
                    parent_tissue=OncoTreeAPI_output_fill0.loc[index2,'level_1']
                    count=count+1
                    parent_tissue_list.append(parent_tissue)
                    print(parent_tissue)
                
    print(parent_tissue_list)
                
    if (len(parent_tissue_list) != 0):
        unique_parent_list=[]
        for j in (parent_tissue_list):
            if j not in unique_parent_list:
                unique_parent_list.append(j)
                uniquecount=uniquecount+1
    
    colindexuniquecount=COSMIC_classification_OT.columns.get_loc('unique_count_parents_in_OncoTree')
    COSMIC_classification_OT.at[index1,colindexuniquecount]=uniquecount          
    colindexlist=COSMIC_classification_OT.columns.get_loc('parent_tissue_code_list')
    COSMIC_classification_OT.at[index1,colindexlist]=str(parent_tissue_list)
    colindexcount=COSMIC_classification_OT.columns.get_loc('count_parents_in_OncoTree')
    COSMIC_classification_OT.at[index1,colindexcount]=count
    colindex=COSMIC_classification_OT.columns.get_loc('parent_tissue_code')
    COSMIC_classification_OT.at[index1,colindex]=parent_tissue
    parent_tissue="Cancer Type Detailed not in OncoTree API file"
    #print(row1)
end_time_block2=time.asctime(time.localtime())
print(end_time_block2)

## Combine somatic data from cBioPortal and COSMIC

In [190]:
## add to cbio and compare gene wise split by cancer tissue type:

start_time_block2=time.asctime(time.localtime())
print(start_time_block2)
#initialize dfs to collect info gene-wise
#1-VEP categories that are filtered out (expected to be filtered by cbio so removing from analysis, but outputting to this df for counts and QC)
VEP_categories_filtered_45TSGs=pd.DataFrame()
#2-all TSG, including splice region, all output stats to this df gene-wise
allTSG_df=pd.read_csv("OncoTree_parenttissuenames_11-25-24.txt", sep="\t").set_index("level_1")
allTSG_df_cbio=pd.read_csv("OncoTree_parenttissuenames_11-25-24.txt", sep="\t").set_index("level_1")

#3-all TSG, excluding splice region, all output stats to this df gene-wise
allTSG_df_nospliceregion=pd.DataFrame()

FLAG1_count=FLAG5_count=FLAGFISHER_count=0

parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)
# loading VEP consequence titles (category names) to use as index to join cbio and clinvar processed dfs:
VEP_calc_consequences=pd.read_csv("VEP_calculated_consequences_8-26-24.txt", sep="\t", header=None) #edited to add 'splice_site_variant' on 8-26-24


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

Hypermutators=pd.read_csv("cbio_sampleinfo_tmb_gt10_12-14-23.txt", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        
        clinvar_cbio_tocompare=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_cbio_tocompare_8-16-24.txt", sep="\t")
        Variation_interpretation_toexclude = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic", "Conflicting interpretations of pathogenicity greater than or equal to 75"]
        clinvar_cbio_tocompare_VUSLBB=clinvar_cbio_tocompare[~clinvar_cbio_tocompare["GL_first_Description"].isin(Variation_interpretation_toexclude)]
        #print("shared count")
        #print(len(clinvar_cbio_tocompare))
        #print("shared that are VUSLBB")
        #print(len(clinvar_cbio_tocompare_VUSLBB))
        cbio_VEP=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_OTtissue_10-24-24.txt", sep="\t")
        #print("cbio VEP len")
        cbioCount1=len(cbio_VEP)
        #print(len(cbio_VEP))
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"] #8-23-24 #, ">=50bp_indel"]
        cbio_VEP=cbio_VEP[~cbio_VEP["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        #print("cbio VEP len after filter VEP categories")
        #print(len(cbio_VEP))
        cbioCount2=len(cbio_VEP)
        Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index
        #print("with hypermutators count QC same as above")
        totalbeforehypermutatorfilter=len(cbio_VEP)
        #print(len(cbio_VEP))
        cbio_VEP_nohypermutator=cbio_VEP[~cbio_VEP["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        cbio_VEP=cbio_VEP_nohypermutator
        #print("without hypermutators count")
        #print(len(cbio_VEP))
        #print("% vars lost to hypermutator filter=#total-#afterfilter/#total")
        #print(((totalbeforehypermutatorfilter-len(cbio_VEP))*100)/totalbeforehypermutatorfilter)
        removetheseVUSLBB=clinvar_cbio_tocompare_VUSLBB["@id"]
        cbio_VEP_noVUSLBB=cbio_VEP[~cbio_VEP["@id"].isin(removetheseVUSLBB)]
        #print("without VUSLBB")
        #print(len(cbio_VEP_noVUSLBB))
        #print("# variants lost after VUSLBB filter")
        #print(len(cbio_VEP)-len(cbio_VEP_noVUSLBB))
        #print("% vars lost to VUSLBB filter=#total-#afterfilter/#total")
        #print(((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP))
        cbio_VEP_cancertypeall=cbio_VEP_noVUSLBB
        print(len(cbio_VEP_cancertypeall))
        #cbio=pd.concat([cbio, cbio_VEP_cancertype])
        cbio_VEP_cancertype_uniqueOT=cbio_VEP_cancertypeall[cbio_VEP_cancertypeall["unique_count_parents_in_OncoTree"]==1]
        print(len(cbio_VEP_cancertype_uniqueOT))
        
        cbio_VEP_cancertype_uniqueOT_0tummorAltCount=cbio_VEP_cancertype_uniqueOT[cbio_VEP_cancertype_uniqueOT["tumorAltCount"]==0]
        cbio_VEP_cancertype_uniqueOT_no0tummorAltCount=cbio_VEP_cancertype_uniqueOT.drop(cbio_VEP_cancertype_uniqueOT_0tummorAltCount.index)
        print(len(cbio_VEP_cancertype_uniqueOT_no0tummorAltCount))
        
        folder_name_aslist=[folder_name]
        cosmic=cosmicconcat_OT_parent_annotated_readcsv_renamecolumns[cosmicconcat_OT_parent_annotated_readcsv_renamecolumns["GENE_SYMBOL"].isin(folder_name_aslist)==True]
        print(len(cosmic))
        cosmic_uniqueOT=cosmic[cosmic["unique_count_parents_in_OncoTree"]==1]
        
        somaticconcat=pd.concat([cbio_VEP_cancertype_uniqueOT_no0tummorAltCount,cosmic_uniqueOT])
        print(len(somaticconcat))
        display(cbio_VEP_cancertype_uniqueOT_no0tummorAltCount["parent_tissue_code"].value_counts())
        display(somaticconcat["parent_tissue_code"].value_counts())
        
        #cbio_VEP_cancertype_uniqueOT_no0tummorAltCount["parent_tissue_code"].value_counts().plot(kind="barh")
        #plt.show
        #somaticconcat["parent_tissue_code"].value_counts().plot(kind='barh')
        #plt.show()
        
        cbio_df=pd.DataFrame(cbio_VEP_cancertype_uniqueOT_no0tummorAltCount["parent_tissue_code"].value_counts())
        somaticconcat_df=pd.DataFrame(somaticconcat["parent_tissue_code"].value_counts())
        
        allTSG_df_cbio=allTSG_df_cbio.join(cbio_df, rsuffix=folder_name)
        allTSG_df=allTSG_df.join(somaticconcat_df, rsuffix=folder_name)
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)


Mon Nov 25 18:49:13 2024
131
125
125
471
596


Myeloid (MYELOID)                  77
Lymphoid (LYMPH)                   10
Lung (LUNG)                         9
Kidney (KIDNEY)                     6
Pancreas (PANCREAS)                 4
Bowel (BOWEL)                       4
CNS/Brain (BRAIN)                   3
Esophagus/Stomach (STOMACH)         3
Liver (LIVER)                       3
Bladder/Urinary Tract (BLADDER)     2
Bone (BONE)                         2
Breast (BREAST)                     1
Head and Neck (HEAD_NECK)           1
Name: parent_tissue_code, dtype: int64

Myeloid (MYELOID)                  523
Lymphoid (LYMPH)                    14
Lung (LUNG)                         13
Esophagus/Stomach (STOMACH)          9
Kidney (KIDNEY)                      7
Bowel (BOWEL)                        6
Skin (SKIN)                          6
Pancreas (PANCREAS)                  4
CNS/Brain (BRAIN)                    3
Breast (BREAST)                      3
Liver (LIVER)                        3
Bladder/Urinary Tract (BLADDER)      2
Bone (BONE)                          2
Head and Neck (HEAD_NECK)            1
Name: parent_tissue_code, dtype: int64

Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1
634
634
634
2258
2877


Kidney (KIDNEY)                    603
Pancreas (PANCREAS)                 11
Lung (LUNG)                          5
Uterus (UTERUS)                      3
CNS/Brain (BRAIN)                    2
Bone (BONE)                          1
Bladder/Urinary Tract (BLADDER)      1
Head and Neck (HEAD_NECK)            1
Esophagus/Stomach (STOMACH)          1
Liver (LIVER)                        1
Ovary/Fallopian Tube (OVARY)         1
Breast (BREAST)                      1
Thyroid (THYROID)                    1
Skin (SKIN)                          1
Adrenal Gland (ADRENAL_GLAND)        1
Name: parent_tissue_code, dtype: int64

...

Bowel (BOWEL)                          266
Lymphoid (LYMPH)                       234
Lung (LUNG)                            147
Esophagus/Stomach (STOMACH)             80
Kidney (KIDNEY)                         56
Pancreas (PANCREAS)                     43
Prostate (PROSTATE)                     42
Bladder/Urinary Tract (BLADDER)         40
Skin (SKIN)                             40
Liver (LIVER)                           36
Breast (BREAST)                         31
Biliary Tract (BILIARY_TRACT)           22
Uterus (UTERUS)                         21
Thyroid (THYROID)                       18
Ovary/Fallopian Tube (OVARY)            16
CNS/Brain (BRAIN)                       14
Myeloid (MYELOID)                       11
Ampulla of Vater (AMPULLA_OF_VATER)      8
Head and Neck (HEAD_NECK)                7
Soft Tissue (SOFT_TISSUE)                6
Adrenal Gland (ADRENAL_GLAND)            5
Other (OTHER)                            4
Thymus (THYMUS)                          3
Cervix (CER

Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/ATM
4060
3978
3963
4977
8940


Bowel (BOWEL)                          3443
Lung (LUNG)                             138
Esophagus/Stomach (STOMACH)             101
Prostate (PROSTATE)                      64
Ampulla of Vater (AMPULLA_OF_VATER)      42
Liver (LIVER)                            36
Biliary Tract (BILIARY_TRACT)            35
Pancreas (PANCREAS)                      21
Other (OTHER)                            14
Skin (SKIN)                              11
Breast (BREAST)                          11
Bladder/Urinary Tract (BLADDER)           7
Head and Neck (HEAD_NECK)                 6
Ovary/Fallopian Tube (OVARY)              6
Uterus (UTERUS)                           6
Kidney (KIDNEY)                           5
CNS/Brain (BRAIN)                         5
Thyroid (THYROID)                         5
Cervix (CERVIX)                           2
Bone (BONE)                               2
Soft Tissue (SOFT_TISSUE)                 2
Myeloid (MYELOID)                         1
Name: parent_tissue_code, dtype:

Bowel (BOWEL)                          8119
Esophagus/Stomach (STOMACH)             199
Lung (LUNG)                             184
Prostate (PROSTATE)                      80
Liver (LIVER)                            50
Ampulla of Vater (AMPULLA_OF_VATER)      44
Biliary Tract (BILIARY_TRACT)            43
Skin (SKIN)                              38
Thyroid (THYROID)                        31
Pancreas (PANCREAS)                      29
Head and Neck (HEAD_NECK)                21
Breast (BREAST)                          15
Other (OTHER)                            15
Adrenal Gland (ADRENAL_GLAND)            13
Bladder/Urinary Tract (BLADDER)          12
Uterus (UTERUS)                          12
Ovary/Fallopian Tube (OVARY)             10
CNS/Brain (BRAIN)                         7
Kidney (KIDNEY)                           5
Soft Tissue (SOFT_TISSUE)                 3
Lymphoid (LYMPH)                          3
Cervix (CERVIX)                           2
Bone (BONE)                     

Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/APC
Mon Nov 25 18:49:21 2024


In [192]:
allTSG_df.fillna(0).to_csv("somaticconcat_parentOTcounts_11-25-24.txt", sep="\t")
allTSG_df_cbio.fillna(0).to_csv("somatic_cbioonly_parentOTcounts_11-25-24.txt", sep="\t")

In [ ]:
## made hbar plots in the excel spreadsheets made from .txt files exported here 